# Cài đặt thư viện

Cài đặt các thư viện cần thiết để phân tích mạng và phát hiện cộng đồng gồm:
- NetworkX: xử lý đồ thị
- Infomap: thuật toán phát hiện cộng đồng Infomap
- iGraph và Leiden: phát hiện cộng đồng bằng Leiden Algorithm

In [ ]:
!pip install infomap --quiet
!pip install igraph --quiet
!pip install networkx>=3.2.1 --quiet

In [ ]:
import os
import time
import pandas as pd
import networkx as nx
from google.colab import drive
from infomap import Infomap
import infomap
import igraph as ig

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PATH = "/content/drive/MyDrive/Social Network Analysis/Data/raw/amazon0302.txt"

# Tải dữ liệu mạng

Xây dựng lớp LoadData để đọc danh sách cạnh từ file dữ liệu và tạo đồ thị có hướng (Directed Graph).

In [ ]:
class LoadData:
    def __init__(self, file_path):
        self.file_path = file_path
        self.graph = nx.DiGraph()

    def load(self):
        with open(self.file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith('#'):
                    continue

                parts = line.split()
                if len(parts) == 2:
                    u, v = map(int, parts)
                    self.graph.add_edge(u, v)

        return self.graph

# Khởi tạo đồ thị

Đọc dữ liệu từ file và lưu dưới dạng đối tượng NetworkX để phục vụ các bước phân tích tiếp theo.

In [ ]:
loader = LoadData(PATH)
G = loader.load()

# Đánh giá chất lượng phân cụm

Lớp Evaluator được sử dụng để tính các chỉ số đánh giá kết quả phát hiện cộng đồng

In [ ]:
class Evaluator:
    def __init__(self, graph):
        self.G = graph
        self.num_nodes = graph.number_of_nodes()
        self.num_edges = graph.number_of_edges()

    def evaluate(self, name, communities, runtime):
        try:
            mod = nx.community.modularity(self.G, communities)
        except:
            mod = 0.0

        intra_edges = 0
        for comm in communities:
            subgraph = self.G.subgraph(comm)
            intra_edges += subgraph.number_of_edges()

        cov = intra_edges / self.num_edges if self.num_edges > 0 else 0.0

        total_conductance = 0.0
        total_internal_density = 0.0

        num_comm = len(communities)
        m = self.num_edges
        degrees = dict(self.G.degree())
        total_vol = 2 * m

        for comm in communities:
            comm_nodes = set(comm)
            n_c = len(comm_nodes)

            if n_c == 0:
                continue

            sub_g = self.G.subgraph(comm_nodes)

            internal_edges = sub_g.number_of_edges()
            vol_c = sum(degrees[node] for node in comm_nodes)

            cut_edges = vol_c - (2 * internal_edges)
            vol_not_c = total_vol - vol_c

            # Internal Density
            if n_c > 1:
                den_c = internal_edges / (n_c * (n_c - 1) / 2)
            else:
                den_c = 1.0

            total_internal_density += den_c

            # Conductance
            min_vol = min(vol_c, vol_not_c)
            cond_c = cut_edges / min_vol if min_vol > 0 else 0.0

            total_conductance += cond_c

        avg_conductance = (
            total_conductance / num_comm
            if num_comm > 0 else 0.0
        )

        avg_internal_density = (
            total_internal_density / num_comm
            if num_comm > 0 else 0.0
        )

        return {
            "Algorithm": name,
            "Runtime (s)": round(runtime, 4),
            "Community": num_comm,
            "Modularity": round(mod, 6),
            "Coverage": round(cov, 6),
            "Avg Conductance": round(avg_conductance, 6),
            "Avg Internal Density": round(avg_internal_density, 6)
        }

# Thuật toán Infomap

Áp dụng thuật toán Infomap để phát hiện các cộng đồng trong mạng dựa trên luồng thông tin giữa các nút.

In [ ]:
class Infomap:
    def __init__(self, options="--two-level --silent"):
        self.options = options

    def detect(self, G):
        # Use the actual library class
        im = infomap.Infomap(self.options)

        for u, v in G.edges():
            im.add_link(u, v)

        start_time = time.time()
        im.run()
        runtime = time.time() - start_time

        community_dict = {}
        for node in im.tree:
            if node.is_leaf:
                community_dict.setdefault(node.module_id, []).append(node.node_id)

        communities = [
            frozenset(nodes)
            for nodes in community_dict.values()
        ]

        return communities, runtime

# Thuật toán Label Propagation

Phân cụm mạng bằng cách lan truyền nhãn giữa các nút lân cận cho đến khi đạt trạng thái ổn định.

In [ ]:
class LabelPropagation:
    def detect(self, G):
        start_time = time.time()

        # Label propagation in NetworkX is only implemented for undirected graphs
        # We convert to undirected for this specific step
        if G.is_directed():
            communities = list(
                nx.community.label_propagation_communities(G.to_undirected())
            )
        else:
            communities = list(
                nx.community.label_propagation_communities(G)
            )

        runtime = time.time() - start_time

        return communities, runtime

# Thuật toán Leiden

Sử dụng thuật toán Leiden nhằm tối ưu hóa modularity và tạo ra các cộng đồng có chất lượng cao hơn.

In [ ]:
class Leiden:
    def __init__(self, objective_function="modularity", beta=1.0):
        self.objective_function = objective_function
        self.beta = beta

    def detect(self, G):
        start_time = time.time()

        ig_graph = ig.Graph.from_networkx(G)

        partition = ig_graph.community_leiden(
            objective_function=self.objective_function,
            beta=self.beta
        )

        communities = []
        for cluster in partition:
            actual_nodes = [
                ig_graph.vs[node_idx]["_nx_name"]
                for node_idx in cluster
            ]
            communities.append(frozenset(actual_nodes))

        runtime = time.time() - start_time

        return communities, runtime

In [ ]:
class Benchmark:
    def __init__(self, graph):
        self.G = graph
        self.evaluator = Evaluator(graph)
        self.results = []

    def run(self, algorithms):
        self.results = []

        for algo in algorithms:
            communities, runtime = algo.detect(self.G)

            name = algo.__class__.__name__

            result = self.evaluator.evaluate(
                name,
                communities,
                runtime
            )

            self.results.append(result)

        return self.results

    def to_dataframe(self):
        df = pd.DataFrame(self.results)

        columns_order = [
            "Algorithm",
            "Runtime (s)",
            "Community",
            "Modularity",
            "Coverage",
            "Avg Conductance",
            "Avg Internal Density",
        ]

        return df[columns_order]

# Benchmark các thuật toán

Thực hiện chạy và so sánh các thuật toán:
- Infomap
- Label Propagation
- Leiden

Các chỉ số được sử dụng gồm thời gian thực thi và chất lượng phân cụm.

In [ ]:
benchmark = Benchmark(G)

algorithms = [
    Infomap(),
    LabelPropagation(),
    Leiden()
]

benchmark.run(algorithms)

df_results = benchmark.to_dataframe()

from IPython.display import display
display(df_results)

,Algorithm,Runtime (s),Community,Modularity,Coverage,Avg Conductance,Avg Internal Density
0,Infomap,19.6778,15158,0.809882,0.810016,0.194511,0.692858
1,LabelPropagation,38.3473,24717,0.731250,0.731432,0.320245,1.057075
2,Leiden,6.4945,403,0.920426,0.936278,0.029322,0.635385


# Trực quan hóa cộng đồng

Hiển thị kết quả phân cụm của thuật toán Leiden bằng đồ thị mạng, trong đó mỗi màu đại diện cho một cộng đồng.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import igraph as ig
import colorsys


def visualize_leiden_communities(
    G,
    communities,
    output_img_path,
    layout_name="drl",
    figsize=(25, 25),
    node_size=0.5,
    alpha=0.9,
    dpi=300
):
    """
    Visualize community detection result with highly distinct colors.

    Parameters
    ----------
    G : networkx.Graph
        Input graph.
    communities : list
        List of communities, each community is a set/list of nodes.
    output_img_path : str
        Output image path.
    layout_name : str, default='drl'
        igraph layout algorithm.
    figsize : tuple, default=(25,25)
        Figure size.
    node_size : float, default=0.5
        Node marker size.
    alpha : float, default=0.9
        Node transparency.
    dpi : int, default=300
        Output image resolution.
    """

    # Node -> Community mapping
    node_to_comm = {}
    for comm_id, comm_nodes in enumerate(communities):
        for node in comm_nodes:
            node_to_comm[node] = comm_id

    # Convert to igraph
    ig_graph = ig.Graph.from_networkx(G)

    communities_array = np.array([
        node_to_comm.get(v["_nx_name"], 0)
        for v in ig_graph.vs
    ])

    # Layout
    layout = ig_graph.layout(layout_name)
    coords = np.array(layout.coords)

    # Generate highly distinct colors
    num_communities = len(communities)

    color_list = []
    for i in range(num_communities):
        hue = ((i * 137.508) % 360) / 360.0
        r, g, b = colorsys.hsv_to_rgb(hue, 0.85, 0.85)
        color_list.append([r, g, b])

    color_matrix = np.array(color_list)
    node_colors = color_matrix[communities_array]

    # Plot
    fig, ax = plt.subplots(figsize=figsize, facecolor="white")

    ax.scatter(
        coords[:, 0],
        coords[:, 1],
        c=node_colors,
        s=node_size,
        alpha=alpha,
        edgecolors="none"
    )

    ax.axis("off")

    plt.savefig(
        output_img_path,
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0,
        facecolor=fig.get_facecolor()
    )

    plt.close(fig)

In [ ]:
leiden_communities, _ = algorithms[2].detect(G)

visualize_leiden_communities(
    G,
    communities=leiden_communities,
    output_img_path="/content/drive/MyDrive/Social Network Analysis/asset/original_community.png"
)